# 04 · Job Hunter Agent

**Goal:** Given a user's `ResumeProfile` + `target_role` + `location`, find relevant job postings on the open web, score them against the user's skills, and return a ranked list.

This notebook is an agentic version of what's currently in `backend/app/job_hunter/agent.py`, with three improvements:

1. **Parallel query fan-out.** Instead of running one Tavily query, the planner emits 3-4 complementary queries (general role, senior-adjusted, site-scoped like `site:greenhouse.io`) and all run in one iteration.
2. **Deduplication + filtering** before LLM extraction (saves tokens on junk results).
3. **Skill-overlap match scoring** computed in Python first (reliable, cheap) with the LLM only used for canonicalization and gap callouts.

### LangGraph shape
```
START → plan → search(parallel) → dedup+filter → extract → score → (maybe loop) → END
                                                                ↑
                                                          if count<3 and iter<2
```


## 0. Setup

In [ ]:
# !pip install -q langchain langchain-google-genai langchain-groq langchain-community tavily-python langgraph pydantic python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path("../backend/.env").resolve())
assert os.getenv("GOOGLE_API_KEY")
assert os.getenv("TAVILY_API_KEY")
print("OK")

## 1. Tools

Tavily for search. We'll keep `search_depth="basic"` here (not "advanced") because job boards tend to be content-rich on the landing page; advanced depth doesn't help much and burns more quota.


In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults(
    max_results=5,
    search_depth="basic",
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

## 2. Schemas

In [ ]:
from typing import List, Optional, Literal, TypedDict
from pydantic import BaseModel, Field

class JobMatch(BaseModel):
    title: str
    company: str
    location: str
    remote: bool = False
    seniority: Literal["Intern", "Entry", "Junior", "Mid", "Senior", "Unknown"] = "Unknown"
    match_pct: int = Field(description="0-100 overlap with user's skills")
    matched: List[str] = Field(description="Skills that hit for this role")
    missing: List[str] = Field(description="Required skills the user is missing")
    salary: Optional[str] = None
    posted_days: int = 0
    description: str
    external_url: str

class ScrapedJobs(BaseModel):
    jobs: List[JobMatch]

## 3. State + nodes

In [ ]:
class JobState(TypedDict, total=False):
    target_role: str
    user_skills: List[str]
    location: str
    queries: List[str]
    raw_results: List[dict]
    parsed_jobs: List[JobMatch]
    iteration: int
    max_iter: int

### Planner

The planner is deterministic — no LLM. We synthesize 3 complementary queries. (You can upgrade to an LLM planner if you want adaptive queries by level/company-type, but this is what almost always wins.)


In [ ]:
def node_plan(state: JobState) -> JobState:
    role = state["target_role"]
    loc  = state.get("location") or "remote"
    queries = [
        f'"{role}" jobs {loc} site:linkedin.com/jobs',
        f'{role} hiring {loc} site:greenhouse.io OR site:lever.co',
        f'entry level {role} remote OR {loc}',
    ]
    return {"queries": queries, "iteration": state.get("iteration", 0) + 1,
            "raw_results": state.get("raw_results", []), "parsed_jobs": state.get("parsed_jobs", [])}

### Search (parallel fan-out)

We fire all queries in parallel with `asyncio` via `tavily.ainvoke`. For a plain notebook we use a thread pool since Tavily client is sync-compatible either way.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def node_search(state: JobState) -> JobState:
    def run(q):
        return tavily.invoke({"query": q})
    with ThreadPoolExecutor(max_workers=4) as pool:
        result_sets = list(pool.map(run, state["queries"]))
    flat = [r for rs in result_sets for r in rs]
    return {"raw_results": state.get("raw_results", []) + flat}

### Dedup + filter

Drop duplicates by URL, strip anything obviously not a job post (too short, same-url dupes, unrelated domains).


In [ ]:
KEEP_SUBSTRINGS = ("linkedin.com/jobs", "greenhouse.io", "lever.co", "ashbyhq.com",
                   "careers.", "/jobs/", "/job/", "wellfound.com", "ycombinator.com/jobs")

def node_dedup_filter(state: JobState) -> JobState:
    seen = set()
    out = []
    for r in state.get("raw_results", []):
        url = (r.get("url") or "").lower()
        if not url or url in seen:
            continue
        if not any(s in url for s in KEEP_SUBSTRINGS):
            continue
        if len((r.get("content") or "")) < 120:
            continue
        seen.add(url)
        out.append(r)
    return {"raw_results": out}

### Extract structured jobs with an LLM

We use **Groq (LLaMA-3-70B)** here because (a) it's much faster than Gemini for structured extraction, and (b) leaves the Gemini quota for reasoning-heavy agents (extraction + deep researcher). Fallback to Gemini if Groq rate-limits.


In [ ]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

def get_extractor_llm():
    if os.getenv("GROQ_API_KEY"):
        return ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)
    return ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

extract_prompt = ChatPromptTemplate.from_template('''You are a job-board scraper. Turn RAW SEARCH RESULTS into JobMatch objects.

TARGET ROLE: {target_role}
USER SKILLS: {user_skills}

For each result:
- Extract title, company, location. If location is missing set it to "Unknown".
- Set `remote=true` if the description says remote / fully remote / work from home.
- Standardize `seniority`.
- `matched`: skills from USER SKILLS that appear in the description.
- `missing`: required skills in the description that are NOT in USER SKILLS.
- `match_pct`: round(100 * len(matched) / max(len(matched)+len(missing), 1)).
- Copy URL verbatim into `external_url`.
- If a result isn't a real job posting, skip it.

RAW RESULTS:
{raw}''')

def node_extract(state: JobState) -> JobState:
    llm = get_extractor_llm()
    structured = llm.with_structured_output(ScrapedJobs)
    raw_str = "\n\n---\n\n".join(
        f"URL: {r.get('url')}\n{r.get('content','')[:800]}" for r in state["raw_results"][:15]
    )
    out: ScrapedJobs = (extract_prompt | structured).invoke({
        "target_role": state["target_role"],
        "user_skills": ", ".join(state["user_skills"]),
        "raw": raw_str,
    })
    combined = state.get("parsed_jobs", []) + out.jobs
    return {"parsed_jobs": combined}

### Re-score in Python

LLM-estimated `match_pct` drifts. We recompute it deterministically from the `matched` / `missing` lists.


In [ ]:
def node_rescore(state: JobState) -> JobState:
    user_set = {s.lower() for s in state["user_skills"]}
    jobs = state.get("parsed_jobs", [])
    for j in jobs:
        mset = {s.lower() for s in j.matched}
        miss = {s.lower() for s in j.missing}
        # Reconcile matched against user_set (strip hallucinations).
        j.matched = sorted(mset & user_set)
        j.missing = sorted(miss - user_set)
        denom = max(len(j.matched) + len(j.missing), 1)
        j.match_pct = round(100 * len(j.matched) / denom)
    jobs.sort(key=lambda j: -j.match_pct)
    return {"parsed_jobs": jobs}

### Router — should we loop?

In [ ]:
def should_loop(state: JobState) -> str:
    if state.get("iteration", 0) >= state.get("max_iter", 2):
        return "end"
    if len(state.get("parsed_jobs", [])) < 3:
        return "loop"
    return "end"  

## 4. Build + visualize the graph

In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(JobState)
g.add_node("plan",    node_plan)
g.add_node("search",  node_search)
g.add_node("dedup",   node_dedup_filter)
g.add_node("extract", node_extract)
g.add_node("rescore", node_rescore)

g.add_edge(START, "plan")
g.add_edge("plan", "search")
g.add_edge("search", "dedup")
g.add_edge("dedup", "extract")
g.add_edge("extract", "rescore")
g.add_conditional_edges("rescore", should_loop, {"loop": "plan", "end": END})

job_agent = g.compile()

try:
    from IPython.display import Image, display
    display(Image(job_agent.get_graph().draw_mermaid_png()))
except Exception:
    print(job_agent.get_graph().draw_mermaid())

## 5. Run end-to-end

In [ ]:
USER_SKILLS = ["Python", "pandas", "scikit-learn", "SQL", "Docker", "AWS", "PyTorch", "FastAPI"]

final = job_agent.invoke({
    "target_role": "Machine Learning Engineer",
    "user_skills": USER_SKILLS,
    "location": "Bengaluru",
    "iteration": 0,
    "max_iter": 2,
}, {"recursion_limit": 15})

jobs = final.get("parsed_jobs", [])
print(f"{len(jobs)} jobs found\n")
for j in jobs[:10]:
    print(f"[{j.match_pct:3d}%] {j.title} — {j.company} ({j.location}{' · remote' if j.remote else ''})")
    print(f"        matched={j.matched}  missing={j.missing[:5]}")
    print(f"        {j.external_url}\n")

## 6. Notes for the next engineer

- **Parallelism.** The biggest latency win here is running the N queries concurrently. In FastAPI-land swap the ThreadPool for `asyncio.gather(tavily.ainvoke(q) for q in queries)`.
- **Domain whitelist.** `KEEP_SUBSTRINGS` is the quality gate. Tune it for the markets you care about — e.g. add `naukri.com`, `instahyre.com` for India.
- **Deterministic scoring > LLM scoring.** Always recompute `match_pct` in Python after the LLM. LLMs drift and the number is user-facing.
- **Redis cache.** Key on (target_role, user_skills_hash, location). TTL ~24h since postings age out.
- **Blocked sites.** LinkedIn and Indeed block plain scrapes; Tavily's `search_depth=advanced` is worth trying if you hit empty content fields. Alternatively route LinkedIn queries through a Serp API wrapper later.
- **Seniority.** The extractor is told to standardize but will still drift on JDs that don't state it. Add a regex pass (e.g. `r"\bsenior|staff|principal\b"`) for bonus signal.
